# Saved HTML classifier smoke test

Load an existing local artifact, then classify HTML. No training, optimizer, dataset
loading, downloads, or writes to model files. Run all cells in your training Python
environment (PyTorch, Transformers and BeautifulSoup installed).

The default artifact is `artifacts/phishing-html-classifier-v1`. Change `MODEL_DIR`
to test another saved model. Preprocessing is read from its `metadata.json`;
legacy v0 artifacts without preprocessing metadata use the original raw-HTML parser.
These examples are diagnostics, not a representative accuracy benchmark.

In [1]:
import json
import math
from pathlib import Path

import torch
from bs4 import BeautifulSoup, Comment
from transformers import MarkupLMFeatureExtractor, MarkupLMForSequenceClassification, MarkupLMProcessor

# Relative paths work from the notebook folder or the repository root.
# You can also set an absolute path to an existing saved artifact directory.
MODEL_DIR = Path("artifacts/phishing-html-classifier-v1")
THRESHOLD_OVERRIDE = None  # None uses saved threshold, or 0.5 for an uncalibrated model.

if not MODEL_DIR.is_absolute():
    candidates = [Path.cwd() / MODEL_DIR,
                  Path.cwd() / "ai" / "markuplm" / MODEL_DIR,
                  Path.cwd() / "markuplm" / MODEL_DIR]
    MODEL_DIR = next((p for p in candidates if (p / "config.json").is_file()), candidates[0])
MODEL_DIR = MODEL_DIR.resolve()
if not (MODEL_DIR / "config.json").is_file():
    raise FileNotFoundError(f"No saved model at {MODEL_DIR}. Set MODEL_DIR to your artifact folder.")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Model:", MODEL_DIR)
print("Device:", device)

: 

In [ ]:
metadata_path = MODEL_DIR / "metadata.json"
metadata = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.is_file() else {}
preprocessing = metadata.get("preprocessing")
if preprocessing is not None and "strip_tags" not in preprocessing:
    raise ValueError("This artifact uses another preprocessing schema. Use its matching inference notebook.")

processor = MarkupLMProcessor.from_pretrained(MODEL_DIR, local_files_only=True)
processor.parse_html = False
feature_extractor = MarkupLMFeatureExtractor()
model = MarkupLMForSequenceClassification.from_pretrained(
    MODEL_DIR, local_files_only=True, use_safetensors=True, dtype=torch.float32,
).to(device).eval()

id2label = {int(k): v for k, v in model.config.id2label.items()}
phishing_ids = [k for k, v in id2label.items() if v.lower() in {"phishing", "phish"}]
if model.config.num_labels != 2 or len(phishing_ids) != 1:
    raise ValueError(f"Expected two classes including phishing; got {id2label}")
PHISHING_ID = phishing_ids[0]
BENIGN_ID = 1 - PHISHING_ID
inference = metadata.get("inference", {})
TEMPERATURE = float(inference.get("temperature", 1.0))
THRESHOLD = float(THRESHOLD_OVERRIDE if THRESHOLD_OVERRIDE is not None
                  else inference.get("operating_threshold", 0.5))
if not math.isfinite(TEMPERATURE) or TEMPERATURE <= 0:
    raise ValueError("Invalid temperature in saved metadata")
if not math.isfinite(THRESHOLD) or not 0 <= THRESHOLD <= 1:
    raise ValueError("Threshold must be between 0 and 1")
MAX_TOKENS = int((preprocessing or {}).get("max_tokens", 512))

print("Labels:", id2label)
print("Saved best epoch:", metadata.get("best_epoch", "not recorded"))
print("Preprocessing:", preprocessing or "legacy raw HTML; no cleaning or character limit")
print("Threshold:", THRESHOLD, "| Temperature:", TEMPERATURE)
print("Scores are softmax outputs; confidence is not a guarantee of correctness.")

Loading weights: 100%|██████████| 307/307 [00:00<00:00, 1386.55it/s]


Labels: {0: 'benign', 1: 'phishing'}
Saved best epoch: not recorded
Preprocessing: legacy raw HTML; no cleaning or character limit
Threshold: 0.5 | Temperature: 1.0
Scores are softmax outputs; confidence is not a guarantee of correctness.


In [ ]:
def extract_page(html):
    if not isinstance(html, str):
        raise TypeError("html must be a string")
    if preprocessing is not None:
        soup = BeautifulSoup(html[:int(preprocessing["parse_char_budget"])], "html.parser")
        for element in soup.find_all(preprocessing["strip_tags"]):
            element.decompose()
        if preprocessing.get("strip_comments", False):
            for comment in soup.find_all(string=lambda text: isinstance(text, Comment)):
                comment.extract()
        encoding = feature_extractor(str(soup))
        pairs = [(node.strip(), xpath)
                 for node, xpath in zip(encoding["nodes"][0], encoding["xpaths"][0]) if node.strip()]
        if not pairs:
            raise ValueError("No usable text nodes")
        nodes, xpaths = zip(*pairs)
        return {"nodes": [list(nodes)], "xpaths": [list(xpaths)]}
    encoding = feature_extractor(html)
    if not encoding["nodes"][0]:
        raise ValueError("No usable text nodes")
    return {"nodes": encoding["nodes"], "xpaths": encoding["xpaths"]}


@torch.inference_mode()
def predict_html(html, show_text=False):
    parsed = extract_page(html)
    inputs = processor(
        nodes=parsed["nodes"], xpaths=parsed["xpaths"], truncation=True,
        padding="max_length", max_length=MAX_TOKENS, return_tensors="pt",
    ).to(device)
    logits = model(**inputs).logits
    if not torch.isfinite(logits).all().item():
        raise RuntimeError("Saved model produced NaN/Inf logits")
    # Float64 on CPU also works when the model runs on MPS.
    probs = torch.softmax(logits.cpu().double() / TEMPERATURE, dim=-1)[0]
    score = probs[PHISHING_ID].item()
    result = {
        "prediction": id2label[PHISHING_ID if score >= THRESHOLD else BENIGN_ID],
        "phishing_score": score,
        "benign_score": probs[BENIGN_ID].item(),
        "threshold": THRESHOLD,
        "html_chars": len(html),
        "input_tokens": int(inputs["attention_mask"].sum().item()),
    }
    if show_text:
        result["model_text"] = processor.tokenizer.decode(inputs["input_ids"][0].tolist(), skip_special_tokens=True)
    return result

## Diagnostic HTML files
Edit or add UTF-8 `.html` files in `examples/benign/` and `examples/phishing/`.
The folder determines the expected label; the notebook discovers files automatically.
The five longer examples have been moved here without changing their content.

MarkupLM normally sees at most 512 tokens, regardless of the file's full size.
These diagnostic examples do not replace representative tests of short-page errors.
The phishing fixture uses a reserved `.invalid` form endpoint; nothing is submitted.


In [ ]:
EXAMPLES_DIR = Path("examples")
if not EXAMPLES_DIR.is_absolute():
    candidates = [Path.cwd() / EXAMPLES_DIR,
                  Path.cwd() / "ai" / "markuplm" / EXAMPLES_DIR,
                  Path.cwd() / "markuplm" / EXAMPLES_DIR]
    EXAMPLES_DIR = next((p for p in candidates
                         if (p / "benign").is_dir() and (p / "phishing").is_dir()), candidates[0])
EXAMPLES_DIR = EXAMPLES_DIR.resolve()

# Expected labels come from folders, never from the model's predictions.
example_files = [(path, class_id)
                 for folder, class_id in (("benign", BENIGN_ID), ("phishing", PHISHING_ID))
                 for path in sorted((EXAMPLES_DIR / folder).glob("*.html"))]
if not example_files:
    raise FileNotFoundError(f"No diagnostic HTML found under {EXAMPLES_DIR}")

results = []
for path, expected_id in example_files:
    name = path.relative_to(EXAMPLES_DIR).as_posix()
    expected = id2label[expected_id]
    try:
        html = path.read_text(encoding="utf-8")
        result = predict_html(html)
        result.update(name=name, expected=expected, passed=result["prediction"] == expected)
        print(("PASS" if result["passed"] else "FAIL"), name, json.dumps(result))
    except (OSError, ValueError, RuntimeError) as exc:
        result = {"name": name, "expected": expected, "error": str(exc), "passed": False}
        print("ERROR", name, str(exc))
    results.append(result)
print(f"{sum(r['passed'] for r in results)}/{len(results)} diagnostic examples passed.")


## Test your own HTML file
Your previous custom HTML is preserved in `examples/custom.html`. Edit that file,
or change `HTML_FILE` below to another local UTF-8 file. This file is not included
in the labeled diagnostic suite. No website is fetched and no model is retrained.


In [ ]:
HTML_FILE = EXAMPLES_DIR / "custom.html"
html = HTML_FILE.read_text(encoding="utf-8")
print(json.dumps(predict_html(html, show_text=True), indent=2, ensure_ascii=False))
